In [1]:
import numpy as np
import pandas as pd
from f1winnerprediction import (
	config, 
	io_fastf1,
	mapper
)
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report,roc_auc_score, roc_curve, auc, mean_absolute_error, r2_score, accuracy_score, f1_score
import fastf1
import fastf1.core
import xgboost as xgb
from pprint import pprint

pd.set_option('display.max_columns', None)

fastf1.Cache.enable_cache(config.FASTF1_RAW_CACHE_DIR.as_posix())

# Load sessions from DB

In [2]:
# sessions: dict[int, list[fastf1.core.Session]] = io_fastf1.fetch_race_sessions_cache(use_sessions_cache=False, use_checkpoint=False)
# sessions

# Load sessions from dump files

In [3]:
sessions: dict[int, list[fastf1.core.Session]] = io_fastf1.load_sessions_from_years()
sessions

{2021: [2021 Season Round 1: Bahrain Grand Prix - Race,
  2021 Season Round 2: Emilia Romagna Grand Prix - Race,
  2021 Season Round 3: Portuguese Grand Prix - Race,
  2021 Season Round 4: Spanish Grand Prix - Race,
  2021 Season Round 5: Monaco Grand Prix - Race,
  2021 Season Round 6: Azerbaijan Grand Prix - Race,
  2021 Season Round 7: French Grand Prix - Race,
  2021 Season Round 8: Styrian Grand Prix - Race,
  2021 Season Round 9: Austrian Grand Prix - Race,
  2021 Season Round 10: British Grand Prix - Race,
  2021 Season Round 11: Hungarian Grand Prix - Race,
  2021 Season Round 12: Belgian Grand Prix - Race,
  2021 Season Round 13: Dutch Grand Prix - Race,
  2021 Season Round 14: Italian Grand Prix - Race,
  2021 Season Round 15: Russian Grand Prix - Race,
  2021 Season Round 16: Turkish Grand Prix - Race,
  2021 Season Round 17: United States Grand Prix - Race,
  2021 Season Round 18: Mexico City Grand Prix - Race,
  2021 Season Round 19: São Paulo Grand Prix - Race,
  2021 Sea

In [4]:
driver_mapping = io_fastf1.build_drivers_dict(sessions)
driver_mapping

{'MSC': {'index': 21},
 'SAI': {'index': 19},
 'GAS': {'index': 19},
 'RAI': {'index': 21},
 'TSU': {'index': 19},
 'ALO': {'index': 19},
 'NOR': {'index': 19},
 'LEC': {'index': 19},
 'STR': {'index': 19},
 'RIC': {'index': 17},
 'OCO': {'index': 19},
 'RUS': {'index': 19},
 'HAM': {'index': 19},
 'PER': {'index': 23},
 'VER': {'index': 19},
 'BOT': {'index': 23},
 'MAZ': {'index': 21},
 'VET': {'index': 21},
 'LAT': {'index': 21},
 'GIO': {'index': 21},
 'KUB': {'index': 13},
 'HUL': {'index': 19},
 'ALB': {'index': 19},
 'ZHO': {'index': 23},
 'MAG': {'index': 23},
 'DEV': {'index': 9},
 'PIA': {'index': 19},
 'SAR': {'index': 14},
 'LAW': {'index': 19},
 'BEA': {'index': 19},
 'COL': {'index': 19},
 'DOO': {'index': 5},
 'ANT': {'index': 19},
 'BOR': {'index': 19},
 'HAD': {'index': 19}}

In [5]:
import importlib

importlib.reload(mapper)


<module 'f1winnerprediction.mapper' from '/media/dhiabenhamouda/Dhia/Work/F1WinnerPrediction/src/f1winnerprediction/mapper.py'>

In [5]:
gp_index_map: dict[str, dict[int, int]] = {}
gp_index_map = mapper.map_gp_indices_between_years(sessions)
pprint(gp_index_map)

{'2021-2022': {0: 0,
               1: 3,
               3: 5,
               4: 6,
               5: 7,
               6: 11,
               8: 10,
               9: 9,
               10: 12,
               11: 13,
               12: 14,
               13: 15,
               16: 18,
               17: 19,
               18: 20,
               20: 1,
               21: 21},
 '2022-2023': {0: 0,
               1: 1,
               2: 2,
               4: 4,
               5: 6,
               6: 5,
               7: 3,
               8: 7,
               9: 9,
               10: 8,
               12: 10,
               13: 11,
               14: 12,
               15: 13,
               16: 14,
               17: 15,
               18: 17,
               19: 18,
               20: 19,
               21: 21},
 '2023-2024': {0: 0,
               1: 1,
               2: 2,
               3: 16,
               4: 5,
               5: 7,
               6: 9,
               7: 8,
            

# Prepare laptime data

In [ ]:
years = config.YEARS_TO_FETCH
# Parse year by year
dataset = pd.DataFrame()
for year_first in tqdm(years):
	if (year_first+1) not in years:
		continue
	second_year = year_first + 1
 
	# Parse session by session
	for session_index, first_session in enumerate(sessions[year_first]):
    
		# Check if the GP index mapping exists
		# gp_index_map_key = str(year_first) + "-" + str(second_year)
		gp_index_map_key = f"{year_first}-{second_year}"
  
		if gp_index_map_key not in gp_index_map:
			# print(f"GP: {gp_index_map_key} not found in mapping.")
			continue

		# Check if the session_index exists in the mapping
		# = if the GP was held in both years
		if session_index not in gp_index_map[gp_index_map_key]:
			print(f"Year: {year_first}, GP: {gp_index_map_key} has no mapped index.")
			continue
		
		try:
     
			# Get the mapped index
			mapped_index = gp_index_map[gp_index_map_key][session_index]
			# print(f"Year: {year_first}, GP: {gp_index_map_key}, Mapped Index: {mapped_index}")
			second_session = sessions[second_year][mapped_index]
				
			# Now we have the current session and the corresponding session for next year
			mean_lap_time_first_year = first_session \
   .laps["LapTime"] \
   .dt.total_seconds() \
   .groupby(first_session.laps["Driver"]) \
   .mean() \
   .sort_values().dropna()
			df_mean_lap_time_first_year = mean_lap_time_first_year.reset_index()
	
			mean_lap_time_latter_year = second_session \
   .laps["LapTime"] \
   .dt.total_seconds() \
   .groupby(second_session.laps["Driver"]) \
   .mean() \
   .sort_values().dropna()
			df_mean_lap_time_next_year = mean_lap_time_latter_year.reset_index()	

			# Merge both DataFrames on 'Driver'
			merged_df = pd.merge(df_mean_lap_time_first_year, df_mean_lap_time_next_year, on="Driver", suffixes=('_first_year', '_latter_year'))
			# print(merged_df)
		except Exception as e:
			print(f"An error occurred for Year: {year_first}, GP: {gp_index_map_key}, Mapped Index: {mapped_index}. Error: {e}")
			continue
		
		dataset = pd.concat([dataset, merged_df], ignore_index=True)
  
dataset.sort_values(by=['Driver'], inplace=True)
dataset

 20%|██        | 1/5 [00:00<00:00,  9.57it/s]

Year: 2021, GP: 2021-2022 has no mapped index.
Year: 2021, GP: 2021-2022 has no mapped index.
Year: 2021, GP: 2021-2022 has no mapped index.
Year: 2021, GP: 2021-2022 has no mapped index.
Year: 2021, GP: 2021-2022 has no mapped index.
Year: 2022, GP: 2022-2023 has no mapped index.
Year: 2022, GP: 2022-2023 has no mapped index.


100%|██████████| 5/5 [00:00<00:00, 11.15it/s]

An error occurred for Year: 2024, GP: 2024-2025, Mapped Index: 20. Error: The data you are trying to access has not been loaded yet. See `Session.load`
An error occurred for Year: 2024, GP: 2024-2025, Mapped Index: 21. Error: The data you are trying to access has not been loaded yet. See `Session.load`
An error occurred for Year: 2024, GP: 2024-2025, Mapped Index: 22. Error: The data you are trying to access has not been loaded yet. See `Session.load`
An error occurred for Year: 2024, GP: 2024-2025, Mapped Index: 23. Error: The data you are trying to access has not been loaded yet. See `Session.load`


,Driver,LapTime_first_year,LapTime_latter_year
1063,ALB,84.493620,87.478476
992,ALB,96.972327,95.926667
1124,ALB,96.450885,108.297745
452,ALB,116.506357,114.696227
1112,ALB,72.480914,80.015200
...,...,...,...
934,ZHO,85.580714,90.416364
696,ZHO,81.057848,81.780046
295,ZHO,99.546170,99.019080
632,ZHO,87.850135,85.458526


### Save new dataset

In [8]:
dataset.to_csv(config.LAP_TIMES_DATASET_CSV_PATH, index=False)

### Load dataset

In [6]:
dataset = pd.read_csv(config.LAP_TIMES_DATASET_CSV_PATH)
dataset.head()

,Driver,LapTime_first_year,LapTime_latter_year
0,ALB,114.696227,110.470250
1,ALB,100.465088,98.511214
2,ALB,103.177630,102.145732
3,ALB,81.016153,84.988514
4,ALB,102.143717,100.944320


### Cleaning

In [7]:
clean_dataset = dataset.dropna()
clean_dataset

,Driver,LapTime_first_year,LapTime_latter_year
0,ALB,114.696227,110.470250
1,ALB,100.465088,98.511214
2,ALB,103.177630,102.145732
3,ALB,81.016153,84.988514
4,ALB,102.143717,100.944320
...,...,...,...
1296,ZHO,72.793014,73.689143
1297,ZHO,92.128581,85.764052
1298,ZHO,101.666184,115.406273
1299,ZHO,99.019080,98.048479


### Training

In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

def data_label_split(df_windows: pd.DataFrame):
	X = df_windows.iloc[:, :-1]
	y = df_windows.iloc[:, -1]
	return X, y

def normalize_features(X: pd.DataFrame):
   scaler = StandardScaler()
   X_normalized = scaler.fit_transform(X)
   return X_normalized


In [27]:
data = clean_dataset[["LapTime_first_year", "LapTime_latter_year"]]
X, y = data_label_split(data)
X = normalize_features(X)
X.shape, y.shape

((1244, 1), (1244,))

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
X_train.shape

(995, 1)

In [29]:
params = {'colsample_bytree': 1.0, 
          'learning_rate': 0.05,
          'max_depth': 8, 
          'n_estimators': 250, 
          'subsample': 1.0}
model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [30]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 4.497094735195207
R² Score: 0.6083489636493987


In [32]:
model.save_model(config.FASTF1_MODELS_DIR / "lap_time_mean_model.json")
model.save_model(config.FASTF1_MODELS_DIR / "lap_time_mean_model.ubj")
print("Model saved successfully.")

Model saved successfully.


# Include all laps time, don't average lap time per race

In [15]:
def get_laps(session: fastf1.core.Session) -> pd.DataFrame:
	"""
	Extracts lap time data for all drivers from a given session.

	Args:
		session: A fastf1 session object.

	Returns:
		A pandas DataFrame with Driver, LapNumber, and LapTime in seconds.
	"""
	if not session.laps.empty:
		laps = session.laps.copy()
		laps["LapTime"] = laps["LapTime"].dt.total_seconds()
		return laps[["Driver", "LapNumber", "LapTime"]].dropna()
	return pd.DataFrame()

def get_comparison_data(current_session: fastf1.core.Session, last_year_session: fastf1.core.Session) -> pd.DataFrame:
	"""
	Merges lap times from the same event in two consecutive years.

	Args:
		current_session: The session object for the current year.
		last_year_session: The session object for the previous year.

	Returns:
		A merged DataFrame with lap times from both years aligned by driver and lap number.
	"""
	laps_current = get_laps(current_session)
	laps_last_year = get_laps(last_year_session)

	if laps_current.empty or laps_last_year.empty:
		return pd.DataFrame()

	# Merge data on Driver and LapNumber
	merged_laps = pd.merge(
		laps_last_year, 
		laps_current, 
		on=["Driver", "LapNumber"], 
		suffixes=('_last_year', '_current_year')
	)
	return merged_laps

# --- Main processing loop ---
all_laps_data = []
# Assuming config.YEARS_TO_FETCH is sorted, start from the second year
for year in tqdm(sorted(config.YEARS_TO_FETCH)[1:-1], desc="Processing Years"):
	last_year = year - 1
	
	# Create a quick lookup map for last year's sessions by event name
	last_year_session_map = {
		s.event.EventName: s for s in sessions.get(last_year, [])
	}

	for current_session in sessions.get(year, []):
		# Find the corresponding session from the previous year
		last_year_session = last_year_session_map.get(current_session.event.EventName)
		print(f"Current Year: {year}, Event: {current_session.event.EventName}")
		if last_year_session:
			comparison_df = get_comparison_data(current_session, last_year_session)
			if not comparison_df.empty:
				all_laps_data.append(comparison_df)

# Concatenate all data into a single DataFrame
if all_laps_data:
	training_data = pd.concat(all_laps_data, ignore_index=True)
	print(f"Successfully created training data with {len(training_data)} samples.")
	print(training_data.head())
else:
	print("No matching sessions found to create training data.")
	training_data = pd.DataFrame()

Processing Years:   0%|          | 0/3 [00:00<?, ?it/s]

Current Year: 2022, Event: Bahrain Grand Prix
Current Year: 2022, Event: Saudi Arabian Grand Prix
Current Year: 2022, Event: Australian Grand Prix
Current Year: 2022, Event: Emilia Romagna Grand Prix
Current Year: 2022, Event: Miami Grand Prix
Current Year: 2022, Event: Spanish Grand Prix
Current Year: 2022, Event: Monaco Grand Prix
Current Year: 2022, Event: Azerbaijan Grand Prix
Current Year: 2022, Event: Canadian Grand Prix
Current Year: 2022, Event: British Grand Prix
Current Year: 2022, Event: Austrian Grand Prix
Current Year: 2022, Event: French Grand Prix
Current Year: 2022, Event: Hungarian Grand Prix
Current Year: 2022, Event: Belgian Grand Prix
Current Year: 2022, Event: Dutch Grand Prix


Processing Years:  33%|███▎      | 1/3 [00:00<00:00,  4.00it/s]

Current Year: 2022, Event: Italian Grand Prix
Current Year: 2022, Event: Singapore Grand Prix
Current Year: 2022, Event: Japanese Grand Prix
Current Year: 2022, Event: United States Grand Prix
Current Year: 2022, Event: Mexico City Grand Prix
Current Year: 2022, Event: São Paulo Grand Prix
Current Year: 2022, Event: Abu Dhabi Grand Prix
Current Year: 2023, Event: Bahrain Grand Prix
Current Year: 2023, Event: Saudi Arabian Grand Prix
Current Year: 2023, Event: Australian Grand Prix
Current Year: 2023, Event: Azerbaijan Grand Prix
Current Year: 2023, Event: Miami Grand Prix
Current Year: 2023, Event: Monaco Grand Prix
Current Year: 2023, Event: Spanish Grand Prix
Current Year: 2023, Event: Canadian Grand Prix
Current Year: 2023, Event: Austrian Grand Prix
Current Year: 2023, Event: British Grand Prix
Current Year: 2023, Event: Hungarian Grand Prix
Current Year: 2023, Event: Belgian Grand Prix
Current Year: 2023, Event: Dutch Grand Prix
Current Year: 2023, Event: Italian Grand Prix
Curren

Processing Years:  67%|██████▋   | 2/3 [00:00<00:00,  4.74it/s]

Current Year: 2023, Event: São Paulo Grand Prix
Current Year: 2023, Event: Las Vegas Grand Prix
Current Year: 2023, Event: Abu Dhabi Grand Prix
Current Year: 2024, Event: Bahrain Grand Prix
Current Year: 2024, Event: Saudi Arabian Grand Prix
Current Year: 2024, Event: Australian Grand Prix
Current Year: 2024, Event: Japanese Grand Prix
Current Year: 2024, Event: Chinese Grand Prix
Current Year: 2024, Event: Miami Grand Prix
Current Year: 2024, Event: Emilia Romagna Grand Prix
Current Year: 2024, Event: Monaco Grand Prix
Current Year: 2024, Event: Canadian Grand Prix
Current Year: 2024, Event: Spanish Grand Prix
Current Year: 2024, Event: Austrian Grand Prix
Current Year: 2024, Event: British Grand Prix
Current Year: 2024, Event: Hungarian Grand Prix
Current Year: 2024, Event: Belgian Grand Prix
Current Year: 2024, Event: Dutch Grand Prix
Current Year: 2024, Event: Italian Grand Prix
Current Year: 2024, Event: Azerbaijan Grand Prix
Current Year: 2024, Event: Singapore Grand Prix
Current

Processing Years: 100%|██████████| 3/3 [00:00<00:00,  4.60it/s]

Current Year: 2024, Event: Las Vegas Grand Prix
Current Year: 2024, Event: Qatar Grand Prix
Current Year: 2024, Event: Abu Dhabi Grand Prix
Successfully created training data with 49750 samples.
  Driver  LapNumber  LapTime_last_year  LapTime_current_year
0    HAM        1.0            119.538               101.555
1    HAM        2.0            142.712                99.002
2    HAM        4.0            104.932                98.892
3    HAM        5.0            105.139                98.923
4    HAM        6.0             96.169                99.707


In [16]:
training_data

,Driver,LapNumber,LapTime_last_year,LapTime_current_year
0,HAM,1.0,119.538,101.555
1,HAM,2.0,142.712,99.002
2,HAM,4.0,104.932,98.892
3,HAM,5.0,105.139,98.923
4,HAM,6.0,96.169,99.707
...,...,...,...,...
49745,MAG,53.0,90.445,90.215
49746,MAG,54.0,90.207,91.204
49747,MAG,55.0,90.252,91.958
49748,MAG,56.0,90.429,112.994


In [17]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

def data_label_split(df_windows: pd.DataFrame):
	X = df_windows.iloc[:, :-1]
	y = df_windows.iloc[:, -1]
	return X, y

def normalize_features(X: pd.DataFrame):
   scaler = StandardScaler()
   X_normalized = scaler.fit_transform(X)
   return X_normalized


In [18]:
data = training_data[["LapTime_last_year", "LapTime_current_year"]]
X, y = data_label_split(data)
X = normalize_features(X)
X.shape, y.shape

((49750, 1), (49750,))

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
X_train.shape

(39800, 1)

In [20]:
params = {'colsample_bytree': 1.0, 
          'learning_rate': 0.05,
          'max_depth': 8, 
          'n_estimators': 250, 
          'subsample': 1.0}
model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [21]:
X_test.mean()

np.float64(-0.000587947281292678)

In [22]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 6.751649253935694
R² Score: 0.08674040906915204


In [33]:
type(y_pred)

numpy.ndarray

In [23]:
y_pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 6.751649253935694
R² Score: 0.08674040906915204


In [24]:
pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})

,Actual,Predicted
25532,100.335,103.099411
22265,83.968,86.182320
20753,70.730,72.848808
43923,107.727,104.987961
18763,85.200,102.616356
...,...,...
18907,79.712,87.520805
37175,71.189,109.416306
40295,85.427,87.453102
26621,84.380,85.028999


In [26]:
model.save_model(config.FASTF1_MODELS_DIR / "lap_time_all_model.json")
model.save_model(config.FASTF1_MODELS_DIR / "lap_time_all_model.ubj")
print("Model saved successfully.")

Model saved successfully.
